(course-alzheimer-scalability-heavy-trajectories)=

# Path A - Module 48: Scalability & Heavy Trajectories

What happens when your simulation trajectory is 500 GB? You cannot use `msm.convert()` to load it into memory. Your computer will crash. 

MolSysMT was built with **Big Data** in mind. In this module, you will learn to use **Iterators** to process massive Alzheimer's trajectories without filling your RAM.

In [ ]:
import molsysmt as msm
from molsysmt import systems

# Even with small files, we can use the Scalability API to learn the pattern
villin_traj = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']
villin_topo = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.h5msm']

### 1. The Iterator Pattern
An `Iterator` doesn't load the data; it creates a "pointer" to the file. You can then ask for data in small chunks (e.g., 10 structures at a time).

In [ ]:
# Create an iterator that yields 10 structures per step
iterator = msm.Iterator([villin_topo, villin_traj], selection='molecule_type=="peptide"', 
                        chunk=10, coordinates=True)

for chunk_idx, coords_chunk in enumerate(iterator):
    print(f"Processing chunk {chunk_idx}. Number of structures in this chunk: {len(coords_chunk)}")
    # Perform your analysis on coords_chunk here
    if chunk_idx == 2: break # Stop early for the demo

### 2. High-Level Automation: `ChunkedExecutor`
Writing loops manually is prone to errors. MolSysMT provides the `ChunkedExecutor` which handles the iteration and the accumulation of results for you.

In [ ]:
# Example: Calculate the average Radius of Gyration over a heavy trajectory
# (This is a simplified syntax example of the pattern)
def my_analysis(molsys):
    return msm.structure.get_radius_of_gyration(molsys, heavy_mode='off')

results = []
structure_start = 0
for coordinates in msm.Iterator(
    [villin_topo, villin_traj], chunk=20, coordinates=True
):
    structure_indices = range(structure_start, structure_start + len(coordinates))
    chunk = msm.extract(
        [villin_topo, villin_traj], structure_indices=structure_indices
    )
    results.extend(my_analysis(chunk))
    structure_start += len(coordinates)

print(f"Calculated RG for {len(results)} structures using chunked execution.")

### 3. Memory Efficiency
By using these tools, your memory usage stays constant (flat line) regardless of how long the simulation is. This is the only way to process microsecond-scale simulations on a standard workstation.

--- 

### 🏆 Path A Challenge: The Big Data Engineer

1. Create an **Iterator** for the Villin trajectory with a `chunk` of 5.
2. Inside a loop, calculate the **Center of Mass** of the protein for every chunk.
3. Store only the maximum X-coordinate found in each chunk.
4. Compare the time it takes to initialize an Iterator vs the time it takes to `msm.convert()` the whole file.

You can now handle any file size! In **Module 49**, we will learn how to benchmark the precompiled native kernels and separate loading, preparation, and execution costs.